# PA1 Task 3 - Domain Generalization on PACS (Kaggle, v2)

**The setup.** Same sources (Photo, Art Painting, Cartoon) and same target (Sketch) as Task 2, but now
**nothing may look at Sketch** until Part E. The question: does anything beat plain supervised
training when the target domain is genuinely unavailable?

| Run | What it does | Trained here? |
|---|---|---|
| `erm` | plain supervised training | **no** - it IS the Task 2 Source-only checkpoint |
| `dan_dg` | ERM + pairwise MMD between the three SOURCE domains, lambda_DG = 1 | yes |
| `sam` | ERM + sharpness-aware minimisation, rho = 0.05, two passes per step | yes |
| study | lambda_DG in {0.1, 1, 10}, mirroring Task 2's DAN lambda study | 2 extra runs |
| `diag_dan_dg_unbiased` | DIAGNOSTIC: lambda_DG = 1 with the unbiased MMD estimator | yes, in the **last cell** (Part F), after everything else is saved |

**What changed in v2**
1. **T0 stops the notebook if there is no GPU.** The last run silently used the CPU and took 7 hours.
2. **Nothing is restored.** Every Task 3 run trains in this session. To re-run Task 2 as well, use
   `rerun_task2_task3_kaggle.ipynb` instead: it re-trains both tasks from scratch.
3. **DAN-DG collapse diagnosis** (Part D2, source only), and the estimator-switch diagnostic as the
   **last cell** (Part F). One *Run All* does everything. Because Part F runs after the main results
   are evaluated and saved, it cannot change them, and if it crashes nothing is lost.
   `report()` now prints the feature norm, and its MMD note is corrected: the biased estimator's
   offset is not a harmless constant (see `shared/mmd.py`).

**Datasets to attach** (then check the slugs in T1):
`pa1-repo`, `pa1-task2-b`, `pa1-task3` (**upload the v2 zip as a new version**; it also carries the updated
`shared/mmd.py` and `task2/methods/dan.py`), `pacs-dataset`, `erm-checkpoint`.

**Timing on a GPU (P100/T4):** `dan_dg_lambda0.1` about 20-30 min, `dan_dg_lambda10` about 5-10 min
(it is expected to early-stop at epoch 6 like lambda = 1 did), plus `dan_dg` about 5-10 min and `sam`
35-50 min. Evaluation adds about 10 min. Part F adds 5-30 min: about 5 if
the unbiased run collapses too, up to about 30 if it trains for the full budget. Use **Save Version -> Save & Run
All (Commit)** so a dead tab cannot lose anything.


## Part A - setup

In [ ]:
# T0. Hard GPU check -- runs BEFORE anything else. The previous session trained on the CPU without
# noticing (7 hours). Kaggle: Settings (right panel) -> Accelerator -> GPU P100 or GPU T4 x2, then
# re-run. Set ALLOW_CPU = True only for evaluation-only sessions (Parts D and E are light).
import torch
ALLOW_CPU = False
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
else:
    msg = 'NO GPU -- Settings -> Accelerator -> GPU, then Run All again.'
    if not ALLOW_CPU:
        raise RuntimeError(msg)   # an error stops "Run All" and a Commit
    print('WARNING:', msg, '(continuing because ALLOW_CPU = True)')


In [ ]:
# T1. Assemble the repo and locate the inputs. /kaggle/input is READ-ONLY, so everything is
# copied into /kaggle/working first. Layers are applied in order, later ones overwriting earlier.
import os, glob, shutil, json, math, subprocess, gc
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE = '/kaggle/input/datasets/zylah628'      # <- change if your datasets live elsewhere
REPO = '/kaggle/working/PA1'
OUT  = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

LAYERS = [f'{BASE}/pa1-repo',        # common/, shared/ + shared/splits/, task1/
          f'{BASE}/pa1-task2-b',     # task2/ code (Task 3 inherits its config and engine)
          f'{BASE}/pa1-task3']       # task3/ code + configs

def root_of(d):
    if any(os.path.isdir(f'{d}/{x}') for x in ['task3', 'task2', 'common', 'shared']):
        return d
    subs = [p.rstrip('/') for p in glob.glob(f'{d}/*/') if os.path.isdir(p)]
    return subs[0] if len(subs) == 1 else d

os.makedirs(REPO, exist_ok=True)
for layer in LAYERS:
    assert os.path.isdir(layer), f'Missing dataset: {layer}\nAttached under BASE: {os.listdir(BASE)}'
    shutil.copytree(root_of(layer), REPO, dirs_exist_ok=True)
    print(f'  layered in: {os.path.basename(layer)}')
os.chdir(REPO)

PACS = os.path.dirname(glob.glob(f'{BASE}/pacs-dataset/**/sketch', recursive=True)[0])

# The ERM checkpoint: Task 3's baseline, produced in Task 2. Copied in so the protocol check and the
# evaluation scripts can find it at a stable path.
ck_hits = glob.glob(f'{BASE}/**/best.pt', recursive=True) + glob.glob('/kaggle/input/**/source_only/best.pt', recursive=True)
assert ck_hits, ('No ERM checkpoint found. Upload the Task 2 source_only run folder '
                 '(best.pt + config_resolved.json) as a dataset and attach it.')
ERM_SRC = os.path.dirname(ck_hits[0])
ERM_DIR = 'task2/results/runs_clip0.0/source_only'
os.makedirs(ERM_DIR, exist_ok=True)
for f in os.listdir(ERM_SRC):
    shutil.copy(f'{ERM_SRC}/{f}', f'{ERM_DIR}/{f}')
ERM_CKPT = f'{ERM_DIR}/best.pt'

NEEDED = ['task3/train.py', 'task3/evaluate_sources.py', 'task3/evaluate_sketch.py',
          'task3/configs/dan_dg.yaml', 'task3/configs/sam.yaml', 'shared/engine.py',
          'shared/splits/pacs_sources_seed6304.json', ERM_CKPT, f'{ERM_DIR}/config_resolved.json']
missing = [f for f in NEEDED if not os.path.exists(f)]
print(f'\n  repo      : {sorted(os.listdir(REPO))}')
print(f'  PACS      : {PACS}')
print(f'  ERM ckpt  : {ERM_CKPT}  ({os.path.getsize(ERM_CKPT)/1e6:.1f} MB, copied from {ERM_SRC})')
print(f'  outputs   : {OUT}')
print(f'  MISSING   : {missing if missing else "nothing -- ready"}')
assert not missing


RUNS_DIR = 'task3/results/runs'   # nothing from earlier sessions is restored: every run trains here


In [ ]:
# T2. Match the protocol to the ERM checkpoint. Task 3 inherits task2/configs/base.yaml, so the
# gradient-clipping setting must equal the one the checkpoint was trained with, or train.py will
# refuse to start (by design: ERM and the Task 3 methods must be comparable).
erm_cfg = json.load(open(f'{ERM_DIR}/config_resolved.json'))
erm_clip = erm_cfg.get('grad_clip', 0.0)
print(f'ERM checkpoint was trained with grad_clip = {erm_clip}')
os.system(f"sed -i 's/^grad_clip: .*/grad_clip: {erm_clip}          # matched to the ERM checkpoint/' task2/configs/base.yaml")
os.system("grep '^grad_clip' task2/configs/base.yaml")

from common.io import load_config
t3 = load_config('task3/configs/base.yaml')
KEYS = ['seed', 'num_workers', 'per_domain_batch', 'lr', 'weight_decay', 'grad_clip', 'max_epochs', 'patience']
chk = pd.DataFrame([{'setting': k, 'ERM checkpoint': erm_cfg.get(k), 'Task 3': t3.get(k),
                     'match': 'yes' if erm_cfg.get(k) == t3.get(k) else 'NO'} for k in KEYS])
display(chk)
erm_sum = json.load(open(f'{ERM_DIR}/summary.json'))
print(f"ERM (loaded, never retrained): best mean source-val macro-F1 = {erm_sum['best_mean_macro_f1']:.2f}")
bad = chk[chk['match'] == 'NO']
print('Protocol matches -- safe to train.' if bad.empty else f'MISMATCH in {list(bad.setting)} -- fix before training.')

In [ ]:
# T3. Helpers. report() prints a health summary, saves the curves, and returns a summary row.
CHANCE_LOSS = math.log(7)          # 1.946 = cross-entropy of uniform guessing over 7 classes
PRIOR_LOSS = 1.913                 # cross-entropy of always predicting the source class FREQUENCIES:
                                   # a model whose features carry no class information ends up here
RUNS_DIR = 'task3/results/runs'

def run(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Command failed (exit code {p.returncode}): {cmd}')

def report(run_name):
    """What to look for:
       loss_cls  must end far below 1.95 (chance for 7 classes)
       loss_mmd  DAN-DG's pairwise source alignment penalty. With the biased estimator and 8 vs 8
                 samples it cannot go below ~0.47 even for IDENTICAL domains, so a value near 0.47
                 means 'already as aligned as this estimator can show'. The offset is not harmless:
                 its gradient keeps pulling features smaller (shared/mmd.py). The unbiased
                 diagnostic run instead hovers around 0 and can be slightly negative.
       feat_norm mean L2 norm of the 512-d features (DAN-DG runs trained with the v2 code only).
                 A steady fall together with loss_cls stuck near 1.9 is the collapse signature.
       loss_sam  SAM's loss at the perturbed point theta+eps; the gap above loss_cls is the local
                 sharpness the optimiser actually sees, and it should shrink if SAM is working"""
    d = f'{RUNS_DIR}/{run_name}'
    s, hist = json.load(open(f'{d}/summary.json')), json.load(open(f'{d}/history.json'))
    ep = [h['epoch'] for h in hist]; f1 = [h['val_mean_macro_f1'] for h in hist]
    cls = [h['loss_cls'] for h in hist]
    best_ep = ep[int(pd.Series(f1).idxmax())]
    print(f'\n=== {run_name} ===')
    print(f'  epochs run              : {s["epochs_run"]} of {s["config"]["max_epochs"]}'
          f' ({"early stop" if s["epochs_run"] < s["config"]["max_epochs"] else "full budget"})')
    print(f'  best source-val macro-F1: {s["best_mean_macro_f1"]:.2f} (epoch {best_ep}), last epoch {f1[-1]:.2f}')
    print(f'  classification loss     : {cls[0]:.3f} -> {cls[-1]:.3f}  (min {min(cls):.3f}; 1.95 = chance)')
    if 'loss_mmd' in hist[0]:
        print(f'  source-pair MMD penalty : {hist[0]["loss_mmd"]:.3f} -> {hist[-1]["loss_mmd"]:.3f}   (trend only)')
    if 'loss_sam' in hist[0]:
        g0, g1 = hist[0]['loss_sam'] - hist[0]['loss_cls'], hist[-1]['loss_sam'] - hist[-1]['loss_cls']
        print(f'  loss at theta+eps (SAM) : {hist[0]["loss_sam"]:.3f} -> {hist[-1]["loss_sam"]:.3f}'
              f'   (gap above loss_cls: {g0:.3f} -> {g1:.3f})')
    if 'feat_norm' in hist[0]:
        fn = [h['feat_norm'] for h in hist]
        print(f'  feature norm |F(x)|     : {fn[0]:.3f} -> {fn[-1]:.3f}   (min {min(fn):.3f})')
    gn = [h.get('grad_norm', float('nan')) for h in hist]
    print(f'  max gradient norm       : {max(gn):.3g}')
    degraded = f1[-1] < s['best_mean_macro_f1'] - 10 or cls[-1] > max(0.5, 3 * min(cls))
    v = ('COLLAPSED (never learned)' if min(cls) > 1.6 or s['best_mean_macro_f1'] < 50 else
         'DEGRADED (learned, then diverged)' if degraded else
         'STABLE' if s['best_mean_macro_f1'] >= 90 else 'WEAK (stable but low source F1)')
    print(f'  VERDICT: {v}')

    panels = [('loss_cls', 'classification loss', CHANCE_LOSS)]
    if 'loss_mmd' in hist[0]: panels.append(('loss_mmd', 'source-pair MMD', None))
    if 'loss_sam' in hist[0]: panels.append(('loss_sam', 'loss at theta+eps', None))
    if 'feat_norm' in hist[0]: panels.append(('feat_norm', 'feature norm', None))
    panels += [('val_mean_macro_f1', 'mean source-val macro-F1', None), ('grad_norm', 'gradient norm', None)]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 2.6))
    for ax, (k, title, line) in zip(axes, panels):
        y = [h.get(k, float('nan')) for h in hist]
        ax.plot(ep, y, marker='.')
        if line: ax.axhline(line, color='r', ls=':', lw=1)
        ax.set_title(f'{run_name}: {title}', fontsize=8); ax.set_xlabel('epoch'); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f'{OUT}/curves_{run_name}.png', dpi=140); plt.show(); plt.close(fig)

    return {'run': run_name, 'epochs': s['epochs_run'], 'best_epoch': best_ep,
            'best_src_val_f1': round(s['best_mean_macro_f1'], 2), 'last_src_val_f1': round(f1[-1], 2),
            'min_loss_cls': round(min(cls), 3), 'final_loss_cls': round(cls[-1], 3),
            'final_loss_mmd': round(hist[-1].get('loss_mmd', float('nan')), 3),
            'final_loss_sam': round(hist[-1].get('loss_sam', float('nan')), 3), 'verdict': v}

def save_tables(rows, name='training_summary.csv'):
    ep_rows = []
    for r in (sorted(os.listdir(RUNS_DIR)) if os.path.isdir(RUNS_DIR) else []):
        h = f'{RUNS_DIR}/{r}/history.json'
        if os.path.exists(h):
            ep_rows += [{'run': r, **e} for e in json.load(open(h))]
    if ep_rows: pd.DataFrame(ep_rows).to_csv(f'{OUT}/epochs.csv', index=False)
    if rows: pd.DataFrame(rows).drop_duplicates(subset='run', keep='last').to_csv(f'{OUT}/{name}', index=False)
    print(f'  saved -> {OUT}/epochs.csv ({len(ep_rows)} rows), {OUT}/{name} ({len(rows)} rows)')

print('Helpers ready.  RUNS_DIR =', RUNS_DIR)

In [ ]:
# T4. Data check. The splits shipped with the repo and must match this PACS copy image for image.
EXPECTED = {'photo': 1670, 'art_painting': 2048, 'cartoon': 2344, 'sketch': 3929}
for d, n in EXPECTED.items():
    got = len([p for p in glob.glob(f'{PACS}/{d}/*/*') if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {d:13s} {got:5d}  (expected {n})' + ('' if got == n else '   <-- MISMATCH, stop here'))
src = json.load(open('shared/splits/pacs_sources_seed6304.json'))['domains']
tr = sum(len(s['train']) for s in src.values())
for d, s in src.items():
    print(f'  {d:13s} train={len(s["train"]):5d}  val={len(s["val"]):4d}')
print(f'\n  one epoch = {tr} // 24 = {tr // 24} updates; 30-epoch budget = {30 * (tr // 24)} updates')
print('  the Sketch list exists on disk but NO Task 3 training or source diagnostic opens it')
import torch
print('\n  torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- set Accelerator to GPU')

## Part B - training

**One update:** 8 photo + 8 art + 8 cartoon images, cross-entropy over all 24, plus the method's own
term. Everything else is Task 2's protocol unchanged: AdamW 1e-4, frozen BatchNorm statistics, at
most 30 epochs, patience 5, checkpoint on mean source-validation macro-F1.

**Before running,** write down what you expect increasing lambda_DG to do to source performance, to
source-domain separability, and to Sketch. You may not swap the main setting for a post-hoc winner.

In [ ]:
# T5. Run list. ERM is not trained: it is the checkpoint copied in at T1.
STUDY = 'dan_dg'      # 'dan_dg' -> lambda_DG in {0.1, 1, 10}  (directly comparable with Task 2's DAN study)
                      # 'sam'    -> rho in {0.01, 0.05, 0.1}
COMMON = f'--set data_root={PACS} erm_checkpoint={ERM_CKPT}'
RUN_LIST = [('dan_dg', 'dan_dg'), ('sam', 'sam')]
RUN_LIST += ([('study_dan_dg_lambda0.1', 'dan_dg_lambda0.1'), ('study_dan_dg_lambda10', 'dan_dg_lambda10')]
             if STUDY == 'dan_dg' else
             [('study_sam_rho0.01', 'sam_rho0.01'), ('study_sam_rho0.1', 'sam_rho0.1')])

summary_rows = []
def train(config, run_name):
    if os.path.exists(f'{RUNS_DIR}/{run_name}/summary.json'):
        print(f'{run_name}: already finished -- skipping')
    else:
        run(f'python -m task3.train --config task3/configs/{config}.yaml {COMMON}')
    summary_rows.append(report(run_name))
    save_tables(summary_rows)

print(f'Controlled study: {STUDY}.  Runs to train (ERM is reused, not trained):')
for c, r in RUN_LIST:
    print(f'  {"done   " if os.path.exists(f"{RUNS_DIR}/{r}/summary.json") else "pending"}  {r:18s} (task3/configs/{c}.yaml)')
print('  (the unbiased-MMD diagnostic is trained in the LAST cell, after everything above is saved)')

In [ ]:
# T6. DAN-DG: ERM loss + (lambda_DG/3) * sum of MMD^2 over the three SOURCE pairs
# (photo-art, photo-cartoon, art-cartoon), each estimated from 8 + 8 features with a per-pair
# median bandwidth. No Sketch image is involved. The bet: features that look the same across the
# three domains you CAN see will also suit a fourth you cannot.
# Expect VERDICT: COLLAPSED, like the original run.
train('dan_dg', 'dan_dg')

In [ ]:
# T7. SAM: rho = 0.05, non-adaptive. Each update does two forward/backward passes -- one to find
# the uphill direction, one to take the gradient there -- then steps from the ORIGINAL weights with
# that second gradient. Roughly twice the compute per step. Expect loss_sam slightly above loss_cls,
# with the gap shrinking if the solution really is flattening.
train('sam', 'sam')

In [ ]:
# T8. The controlled study: the two extra settings (the middle one is the main run above).
# Before running, write down your expectation. lambda = 0.1 should train normally (it reached 92 F1 in
# 2 epochs on the CPU). lambda = 10 has 10x the pull that collapsed lambda = 1, so it is expected to
# collapse the same way (loss_cls ~1.9, F1 5.07). If it does, that is a result, not a bug.
for c, r in RUN_LIST[2:]:
    train(c, r)

In [ ]:
# T9. All Task 3 runs side by side, plus the ERM reference. Source information only.
rows = list(pd.DataFrame(summary_rows).drop_duplicates(subset='run', keep='last').to_dict('records'))
rows.append({'run': 'erm (Task 2 checkpoint)', 'epochs': erm_sum['epochs_run'],
             'best_src_val_f1': round(erm_sum['best_mean_macro_f1'], 2), 'verdict': 'loaded, not retrained'})
tbl = pd.DataFrame(rows)
display(tbl)
tbl.to_csv(f'{OUT}/training_summary.csv', index=False)
print('saved ->', f'{OUT}/training_summary.csv')

## Part D - source-side evaluation (still no Sketch)

This is what makes the protocol checkable rather than merely asserted: read it fully, then run Part E.

In [ ]:
# T10. SOURCE-SIDE evaluation. Still no Sketch. This reports, for ERM / DAN-DG / SAM:
#   * accuracy and macro-F1 per source domain, plus the MEAN and the WORST domain
#   * source-domain separability: a 3-way probe (photo vs art vs cartoon) on frozen features,
#     balanced, seed-6304 70/30 split, logistic regression C=1. Chance is 33.3%; lower means the
#     three sources are harder to tell apart, which is what DAN-DG is trying to achieve.
#   * the sharpness proxy: one ascent step of radius 0.05 on a fixed 96-image validation batch
#     (32 per source, eval mode) -- how much the loss rises. Lower = locally flatter.
run(f'python -m task3.evaluate_sources {COMMON}')
run(f'python -m task3.evaluate_sources --study {STUDY} {COMMON}')

F3 = 'task3/results/final'
ss = pd.read_csv(f'{F3}/source_side.csv')
display(ss.round(3))
for _, r in ss.iterrows():
    print(f'  {r.method:10s} mean {r.mean_acc:6.2f}% | worst {r.worst_acc:6.2f}% '
          f'| source separability {r.src_domain_sep:6.2f}% (33.3 = chance) | sharpness {r.sharpness:+.4f}')
print('\nThree questions answerable WITHOUT Sketch:')
print('  1. did DAN-DG actually lower source separability relative to ERM?')
print('  2. did SAM actually lower the sharpness proxy?')
print('  3. does the mean hide a weak domain (compare mean with worst)?')
for f in glob.glob(f'{F3}/*.csv') + glob.glob(f'{F3}/*.json'):
    shutil.copy(f, OUT)
print('\ncopied source-side results to', OUT)

## Part D2 - why did DAN-DG collapse? (still no Sketch)

`task3/diagnose_collapse.py` loads every Task 3 checkpoint and measures, on the source **validation** sets:
which class it predicts most often, the feature norm, a 7-way **class** probe on frozen features (the
counterpart of the domain probe), and the 8-vs-8 MMD with and without the estimator offset. It answers
RQ2's "evidence that alignment removed class-discriminative information" directly.


In [ ]:
# T10b. Collapse diagnostics -- source validation only.
run(f'python -m task3.diagnose_collapse {COMMON}')
cd = pd.read_csv('task3/results/final/collapse_diagnostics.csv')
display(cd.round(3))
print('''
How to read it
  top_pred_share ~100%        the model gives one class to everything (the head collapsed)
  class_probe high (>80%)     the FEATURES still separate the classes -- the head never learned to read
                              them: an optimisation failure
  class_probe near 14.3%      the backbone itself erased class information: alignment destroyed it
  feat_norm far below ERM's   supports the shrinking explanation (biased-MMD pull)
  mmd_biased - floor          the part of the logged MMD that is a real domain difference; for a healthy
                              model the rest is estimator offset
  domain_sep near 33.3%       sources indistinguishable -- read it TOGETHER with class_probe: low domain
                              separability is only good news if class_probe stays high
''')
erm_n = cd.loc[cd.run == 'erm', 'feat_norm']
for _, r in cd.iterrows():
    rel = f'{r.feat_norm / erm_n.iloc[0]:.2f}x ERM' if len(erm_n) else ''
    print(f'  {r.run:22s} F1 {r.mean_f1:6.2f} | {r.top_pred_share:5.1f}% -> {r.top_pred:8s} | |F| {rel:10s} '
          f'| class probe {r.class_probe:5.1f}% | domain sep {r.domain_sep:5.1f}% '
          f'| MMD excess over floor {r.mmd_biased_8v8 - r.floor_biased_8v8:+.3f}')
shutil.copy('task3/results/final/collapse_diagnostics.csv', OUT)
print('\ncopied ->', f'{OUT}/collapse_diagnostics.csv')


## Part E - Sketch evaluation (the only Sketch access)

In [ ]:
# T11. SKETCH evaluation -- the only Sketch access in all of Task 3. Everything above is frozen.
run(f'python -m task3.evaluate_sketch {COMMON}')
run(f'python -m task3.evaluate_sketch --study {STUDY} {COMMON}')

main = pd.read_csv(f'{F3}/table_main.csv')
display(main.round(2))
erm_row = main[main.method == 'erm'].iloc[0]
print('\n================ TASK 3 MAIN COMPARISON ================')
for _, r in main.iterrows():
    tag = 'baseline' if r.method == 'erm' else f'{r.sketch_acc - erm_row.sketch_acc:+.2f} pts vs ERM'
    print(f'  {r.method:10s} source mean {r.mean_f1:6.2f} F1 / worst {r.worst_f1:6.2f} | Sketch {r.sketch_acc:6.2f}% '
          f'| src-sep {r.src_domain_sep:6.2f}% | sharpness {r.sharpness:+.4f}   ({tag})')
print('\n  RQ1: how well did mean-source and worst-source predict this Sketch ranking?')
print('  RQ2: did lower SOURCE separability come with better Sketch accuracy?')
print('  RQ3: does the sharpness ranking agree with the Sketch ranking?')
for f in glob.glob(f'{F3}/*.csv') + glob.glob(f'{F3}/*.json'):
    shutil.copy(f, OUT)

In [ ]:
# T12. The study, the per-class picture, and the Task 2 vs Task 3 comparison (RQ4).
st = pd.read_csv(f'{F3}/table_study_{STUDY}.csv')
display(st.round(3))
for _, r in st.iterrows():
    print(f'  {r.setting:12s} source mean {r.mean_f1:6.2f} F1 | src-sep {r.src_domain_sep:6.2f}% '
          f'| sharpness {r.sharpness:+.4f} | Sketch {r.sketch_acc:6.2f}%')
print(f'\n  chosen WITHOUT Sketch (best mean source F1): {st.loc[st.mean_f1.idxmax()].setting}')
print(f'  best on Sketch, known only now             : {st.loc[st.sketch_acc.idxmax()].setting}')
print('  the main comparison keeps lambda_DG = 1 / rho = 0.05 regardless of what won here\n')

pc = pd.read_csv(f'{F3}/per_class_sketch_acc.csv', index_col=0)
print('Per-class Sketch accuracy (%):'); display(pc.round(2))
print('Change versus ERM (percentage points):')
display(pc.drop(columns=['erm']).sub(pc['erm'], axis=0).round(2))
ana = json.load(open(f'{F3}/class_analysis.json'))
for m, a in ana.items():
    if not isinstance(a, dict) or 'largest_gain' not in a: continue
    g, d = a['largest_gain'], a['largest_drop']
    print(f'\n{m}:  gain {g["class"]} {g["delta_pp"]:+.1f} pts | drop {d["class"]} {d["delta_pp"]:+.1f} pts -> '
          + ', '.join(f'{c["pred"]} ({c["share_%"]:.0f}%)' for c in d['confusions_after'][:2]))
print('\nSketch has only 80 house and 160 person images, so those classes move in large jumps.\n')

print('=== RQ4: what did unlabelled Sketch actually buy in Task 2? ===')
print(open(f'{F3}/task2_vs_task3_overall.json').read())
print('Same MMD penalty and kernels; Task 2 DAN aligned source-to-target, Task 3 DAN-DG aligned')
print('source-to-source. Different pairs, different sample sizes per estimate, one seed each --')
print('suggestive, not a clean ablation of target access.')

## Part S - save

In [ ]:
# T13. Save everything. Run this even if a cell above failed.
for p in glob.glob(f'{F3}/figures/*.png'):
    shutil.copy(p, OUT)
os.system(f'cd {REPO} && zip -qr {OUT}/task3_full.zip task3/results')                 # with checkpoints
os.system(f'cd {REPO} && zip -qr {OUT}/task3_light.zip task3/results -x "*.pt"')      # without
for f in sorted(os.listdir(OUT)):
    print(f'{os.path.getsize(f"{OUT}/{f}")/1e6:8.2f} MB  outputs/{f}')
print('\nDownload task3_light.zip + the CSVs + the PNGs; keep task3_full.zip only if you want the checkpoints.')

## Notes
- **ERM is never retrained.** If you retrain Source-only in Task 2, Task 3's comparison changes and every run here must be redone.
- **`task3/train.py` refuses a protocol mismatch** against the ERM checkpoint. If it stops you, fix `task2/configs/base.yaml` rather than bypassing the check.
- **Kaggle hardware differs from Colab's,** so these runs are not bit-comparable with Task 2's. Say in the report that training was split across two machines.
- **Expected verdicts:** `dan_dg` (lambda = 1) and `dan_dg_lambda10` COLLAPSED; `sam` and `dan_dg_lambda0.1` STABLE.
  If anything else is flagged COLLAPSED or DEGRADED, send its report to Claude before interpreting the results.
- **DAN-DG at lambda_DG = 1 collapsed, and the main row stays that way.** The handout fixes lambda_DG = 1 and
  forbids swapping in a post-hoc winner. Report the collapse, show Part D2's diagnosis, and let the lambda study
  show the working regime. The unbiased-estimator run (Part F) is a labelled diagnostic only.
- **Part F runs after the Task 3 Sketch evaluation,** so the main comparison is locked with the biased
  estimator. Report the unbiased run as a diagnostic; switching the main estimator after Sketch has been
  evaluated would be a post-hoc change.
- **Hardware:** ERM was trained on Colab and these runs on a Kaggle GPU. That changes nothing in the protocol;
  it only adds rounding-level noise.


## Part F - the estimator switch (diagnostic, last on purpose)

DAN-DG again at **lambda_DG = 1**, with the same kernels, bandwidth, batches and protocol. Only the MMD
estimator changes: **unbiased**, so no self-comparisons. That means no ~0.47 floor and no pull that keeps
shrinking the features. Source validation only; it is never evaluated on Sketch and never replaces the
main `dan_dg` row.

| If the unbiased run... | it means |
|---|---|
| trains (source F1 around 90, `loss_cls` well below 1.9) | the biased estimator's small-sample pull is the most likely cause of the collapse |
| collapses the same way (F1 about 5, `loss_cls` about 1.92) | the estimator is not the cause; the MMD gradient at 8 vs 8 is too strong or noisy either way |


In [ ]:
# T14. LAST CELL -- the estimator switch, as a labelled DIAGNOSTIC.
# Everything above (training, source-side evaluation, Sketch evaluation, saving) is already finished
# and saved, so this cell cannot change any main result. If it crashes, nothing above is lost.
# Expect loss_mmd around 0 (it can be slightly negative), not ~0.47: the unbiased estimate has no floor.
RUN_UNBIASED_DIAG = True        # set to False to skip this cell
DIAG = 'diag_dan_dg_unbiased'

if not RUN_UNBIASED_DIAG:
    print('Skipped (RUN_UNBIASED_DIAG = False).')
else:
    # 1) train; prints the usual report + curves, including feat_norm
    train(DIAG, DIAG)

    # 2) re-run the source-only collapse diagnostics so the table now includes the diagnostic run
    run(f'python -m task3.diagnose_collapse {COMMON}')
    cd = pd.read_csv('task3/results/final/collapse_diagnostics.csv').set_index('run')
    display(cd.round(3))

    # 3) side by side: the main lambda = 1 run (biased) vs the diagnostic (unbiased)
    hb = json.load(open(f'{RUNS_DIR}/dan_dg/history.json'))
    hu = json.load(open(f'{RUNS_DIR}/{DIAG}/history.json'))
    sb = json.load(open(f'{RUNS_DIR}/dan_dg/summary.json'))
    su = json.load(open(f'{RUNS_DIR}/{DIAG}/summary.json'))
    def line(label, a, b, fmt='{:.3f}'):
        print(f'  {label:44s}{fmt.format(a):>15s}{fmt.format(b):>17s}')
    print('\n============ lambda_DG = 1: biased (main) vs unbiased (diagnostic) ============')
    print(f'  {"":44s}{"biased/main":>15s}{"unbiased/diag":>17s}')
    line('best mean source-val macro-F1', sb['best_mean_macro_f1'], su['best_mean_macro_f1'], '{:.2f}')
    line('epochs run', sb['epochs_run'], su['epochs_run'], '{:d}')
    line('lowest loss_cls (1.913 = no class info)', min(h['loss_cls'] for h in hb), min(h['loss_cls'] for h in hu))
    line('last loss_mmd (biased floor ~0.47; unbiased ~0)', hb[-1]['loss_mmd'], hu[-1]['loss_mmd'])
    line('max gradient norm', max(h['grad_norm'] for h in hb), max(h['grad_norm'] for h in hu), '{:.3g}')
    if {'dan_dg', DIAG} <= set(cd.index):
        line('class probe on frozen features (%, 14.3 = chance)', cd.loc['dan_dg', 'class_probe'], cd.loc[DIAG, 'class_probe'], '{:.1f}')
        line('feature norm |F(x)| (ERM: see table)', cd.loc['dan_dg', 'feat_norm'], cd.loc[DIAG, 'feat_norm'])
        line('share of images given the top class (%)', cd.loc['dan_dg', 'top_pred_share'], cd.loc[DIAG, 'top_pred_share'], '{:.1f}')
        line('source-domain separability (%, 33.3 = chance)', cd.loc['dan_dg', 'domain_sep'], cd.loc[DIAG, 'domain_sep'], '{:.1f}')

    f1u = su['best_mean_macro_f1']
    print('\nWHAT IT MEANS')
    if f1u >= 80:
        print('  The unbiased run TRAINED. The biased estimator\'s small-sample pull is the most likely cause of the')
        print('  lambda = 1 collapse. The main DAN-DG row stays the biased run; report this as a diagnostic.')
    elif f1u < 50:
        print('  The unbiased run ALSO COLLAPSED. The estimator is not the cause: the MMD gradient from 8-vs-8')
        print('  estimates overwhelms classification either way. Keep the biased results everywhere and report this.')
    else:
        print('  IN BETWEEN: it learned something but not well. Send this cell\'s output to Claude.')
    print('  Either way it is never evaluated on Sketch and never replaces the main dan_dg row.')

    # 4) save again: training table with the ERM row, epochs.csv, diagnostics, and fresh zips
    rows = list(pd.DataFrame(summary_rows).drop_duplicates(subset='run', keep='last').to_dict('records'))
    rows.append({'run': 'erm (Task 2 checkpoint)', 'epochs': erm_sum['epochs_run'],
                 'best_src_val_f1': round(erm_sum['best_mean_macro_f1'], 2), 'verdict': 'loaded, not retrained'})
    pd.DataFrame(rows).to_csv(f'{OUT}/training_summary.csv', index=False)
    shutil.copy('task3/results/final/collapse_diagnostics.csv', OUT)
    os.system(f'cd {REPO} && zip -qr {OUT}/task3_full.zip task3/results')
    os.system(f'cd {REPO} && zip -qr {OUT}/task3_light.zip task3/results -x "*.pt"')
    for f in sorted(os.listdir(OUT)):
        print(f'{os.path.getsize(f"{OUT}/{f}")/1e6:8.2f} MB  outputs/{f}')
    print('\nre-saved: the zips and tables now include the diagnostic run.')
